In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

print("Xarray:", xr.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

# CMIP6 Future Groundwater-Salinity Workflow

This notebook is the experiment guide; it does not treat CMIP6 ocean salinity (`so`) as a groundwater predictor. Supply Person 2's exact feature contract, a target-grid NetCDF with `lat`/`lon`, monthly observed reference features for bias adjustment, training features, and the validated model artifact before running the corresponding stages.

The builder targets SSP2-4.5 (`ssp245`) and SSP5-8.5 (`ssp585`) with 2030 (2021-2040), 2050 (2041-2060), and 2080 (2071-2090) period means. Precipitation `pr` is converted from kg m-2 s-1 to mm day-1; `tas*` is converted from K to degrees C. Regridding uses bilinear interpolation and clips the target coordinates to 5-38N, 65-100E. Products record whether bias correction was applied.

In [6]:
from pathlib import Path
import json
import sys
import xarray as xr

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").is_dir():
    BACKEND_DIR = CURRENT_DIR
elif CURRENT_DIR.name == "notebook" and (CURRENT_DIR.parent / "src").is_dir():
    BACKEND_DIR = CURRENT_DIR.parent
elif (CURRENT_DIR / "Backend" / "src").is_dir():
    BACKEND_DIR = CURRENT_DIR / "Backend"
else:
    raise FileNotFoundError("Run this notebook from the project, Backend, or Backend/notebook directory")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

# Replace only from the approved Person 2 feature contract.
FEATURE_MAP = {}  # exact model feature name: CMIP6 variable ID, e.g. pr/tas/tasmax
TARGET_GRID = BACKEND_DIR / "data" / "processed" / "aquashield_1km_grid.nc"
OBSERVED_REFERENCE = BACKEND_DIR / "data" / "processed" / "observed_monthly_features.nc"
TRAINING_FEATURES = BACKEND_DIR / "data" / "processed" / "training_features.nc"
PREDICTION_ENSEMBLE = BACKEND_DIR / "data" / "future" / "future_prediction_ensemble_ssp245_2030.nc"
OUTPUT_DIR = BACKEND_DIR / "data" / "future"
MODEL_IDS = []  # optional source_id allowlist; empty selects shared available models

print("Backend:", BACKEND_DIR)
print("Feature contract supplied:", bool(FEATURE_MAP))
print("Target grid available:", TARGET_GRID.is_file())

Backend: C:\Users\DELL\Downloads\Salinity project\Backend
Feature contract supplied: False
Target grid available: False


## Catalog selection and climate features

The builder intersects model IDs across every requested variable and, when the catalog provides `member_id`, uses a shared ensemble member across those variables. It requires all three inputs below: the approved feature map, the target grid, and public catalog/network access. It writes six feature files; they are climate features, not salinity predictions.

In [3]:
from src.projection.regrid import build_future_features

if FEATURE_MAP and TARGET_GRID.is_file() and OBSERVED_REFERENCE.is_file():
    feature_paths = build_future_features(
        feature_map=FEATURE_MAP,
        target_grid_path=TARGET_GRID,
        output_dir=OUTPUT_DIR,
        model_ids=MODEL_IDS or None,
        observed_reference_path=OBSERVED_REFERENCE,
    )
    feature_paths
else:
    print("Not run: provide FEATURE_MAP, TARGET_GRID, and monthly OBSERVED_REFERENCE inputs first.")

Not run: provide FEATURE_MAP, TARGET_GRID, and monthly OBSERVED_REFERENCE inputs first.


## Bias adjustment and applicability

When an observed reference dataset is supplied, the builder loads monthly future and matched historical CMIP6 members, regrids each to the target grid, applies monthly additive deltas to temperature-like features and multiplicative ratios to `pr`, then aggregates each target period. The reference file must contain monthly, same-grid variables named exactly as the model features. This is a transparent mean-delta method, not quantile mapping. The applicability stage below compares the corrected future features with Person 2's training features and saves a covariate-shift score, binary mask, bootstrap applicability probability, and bootstrap classification uncertainty.

In [4]:
from src.projection.applicability import assess_applicability

future_feature_file = OUTPUT_DIR / "future_features_ssp245_2030.nc"
applicability_file = OUTPUT_DIR / "applicability_ssp245_2030.nc"
if FEATURE_MAP and TRAINING_FEATURES.is_file() and future_feature_file.is_file():
    with xr.open_dataset(TRAINING_FEATURES) as training, xr.open_dataset(future_feature_file) as future:
        applicability = assess_applicability(training, future, list(FEATURE_MAP))
        applicability.to_netcdf(applicability_file)
    print("Saved:", applicability_file)
else:
    print("Not run: provide training features and a matching corrected future feature product.")

Not run: provide training features and a matching corrected future feature product.


## Validated model inference and uncertainty

No model artifact or feature-schema file is currently present in `Backend/models`, so inference must not be run or fabricated. Person 2 must provide the validated estimator, exact feature order and transformations, prediction target/units, and any model/observation resampling strategy. Inference must use only those exact features, retain independent `climate_model`, `model_draw`, and `observation_draw` axes, and preserve the applicability layers. Predictions outside `applicability_mask == 1` must not be treated as trusted. The uncertainty routine requires at least two samples on each ensemble axis and writes separate standard-deviation layers; it does not combine them into a misleading single scalar.

In [5]:
from src.uncertainty.quantify import calculate_uncertainty

if PREDICTION_ENSEMBLE.is_file():
    standardized_dir = BACKEND_DIR / "data" / "processed"
    final_product = standardized_dir / "future_ssp245_2030.nc"
    calculate_uncertainty(PREDICTION_ENSEMBLE, final_product)
    print("Saved validated future product:", final_product)
else:
    print("Not run: validated Person 2 prediction ensemble is not available.")

Not run: validated Person 2 prediction ensemble is not available.


## API product contract and validation

The API reads `Backend/data/processed/current.nc` and `future_{ssp245|ssp585}_{2030|2050|2080}.nc` by default; set `AQUASHIELD_PRODUCT_DIR` to move these products. Every future file must include `salinity_mean`, `climate_model_spread_sd`, `model_uncertainty_sd`, `data_uncertainty_sd`, `covariate_shift_score`, `applicability_mask`, `applicability_probability`, and `applicability_uncertainty`, plus standardized `lat` and `lon` coordinates. Routes are `/api/v1/maps/{scenario}/{period}`, `/api/v1/location`, `/api/v1/drivers`, `/api/v1/uncertainty`, and `/api/v1/timeseries`. The API returns 404/503 when products or required layers are missing; it does not fall back to the old ocean-salinity file.

## Synthetic end-to-end API demo

For an offline smoke test, generate the isolated files under `Backend/data/demo`. These artifacts are explicitly synthetic and do not replace any validated model or real project products. The demo includes a fitted least-squares model and API-ready current/SSP products on a 1-degree rectangular India-region grid.

In [7]:
from src.demo_data import generate_demo

demo_artifacts = generate_demo(BACKEND_DIR / "data" / "demo")
for artifact_name, artifact_path in demo_artifacts.items():
    print(f"{artifact_name}: {artifact_path}")
print("WARNING: synthetic demo only; not real data or predictions")

output_dir: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo
grid: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\target_grid_india_1deg_demo.nc
fake_observed_reference: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\fake_observed_reference_demo.nc
training_features: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\training\training_features_demo.nc
model: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\models\synthetic_demo_model.json
feature_contract: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\feature_contract_demo.json
products_dir: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\products
product_current_current: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\products\current.nc
product_ssp245_2030: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\products\future_ssp245_2030.nc
product_ssp245_2050: C:\Users\DELL\Downloads\Salinity project\Backend\data\demo\products\future_ssp245_2